In [66]:
import pandas as pd
import numpy as np

In [67]:
# Reads in the two human-labeled datasets
df_1 = pd.read_csv('dataset_1_components/bluesky_posts_tagged.csv', dtype=str)
df_2 = pd.read_csv('dataset_1_components/human_categorized_data.csv', dtype=str)

In [68]:
# Removes all non-labeled posts from the bluesky dataset
x = df_1.dropna(subset='Educator')
y = df_1.dropna(subset='Student')
z = df_1.dropna(subset='Expert/keynote/researchers')

x = x.rename(columns={'Educator':'Label'})
y = y.rename(columns={'Student':'Label'})
z = z.rename(columns={'Expert/keynote/researchers':'Label'})

x['Cat_label'] = np.where(x['Label'] == '1', 'Educator', '0')
y['Cat_label'] = np.where(y['Label'] == '1', 'Student', '0')
z['Cat_label'] = np.where(z['Label'] == '1', 'Expert', '0')

# Drops all columns that are not needed for the final dataset
x.drop(columns=['uri','author_name','author_did','reply','link','Student','Expert/keynote/researchers'], inplace=True)
y.drop(columns=['uri','author_name','author_did','reply','link','Educator','Expert/keynote/researchers'], inplace=True)
z.drop(columns=['uri','author_name','author_did','reply','link','Educator','Student'], inplace=True)

# Order is important here for drop_duplicate function
ds_1 = pd.concat([z, y, x])

# Final labeled bluesky dataset
ds_1.drop_duplicates(subset='text',inplace=True)
ds_1.dropna(subset='text', inplace=True)

In [69]:
# Veryify that no values were fropped (CONFIRMED)
print(x['Label'].value_counts())
print(y['Label'].value_counts())
print(z['Label'].value_counts())
print(ds_1['Label'].value_counts()) 

Label
0    1501
1     512
Name: count, dtype: int64
Label
1    22
Name: count, dtype: int64
Label
1    73
Name: count, dtype: int64
Label
0    1394
1     607
Name: count, dtype: int64


In [70]:
print(x['Cat_label'].value_counts())
print(y['Cat_label'].value_counts())
print(z['Cat_label'].value_counts())
print(ds_1['Cat_label'].value_counts()) 

Cat_label
0           1501
Educator     512
Name: count, dtype: int64
Cat_label
Student    22
Name: count, dtype: int64
Cat_label
Expert    73
Name: count, dtype: int64
Cat_label
0           1394
Educator     512
Expert        73
Student       22
Name: count, dtype: int64


In [71]:
# Reads the papers dataset (Abishethvarman et al. 2023)
ds_2 = pd.read_csv('dataset_1_components/categorized_data.csv', dtype=str)
ds_2.head(5)

,Unnamed: 0,tweet_id,original_text,text,sentiment,tasks,users,technologies,organizations,competencies,job_profiles,category
0,0,1.59783e+18,"@andrew___baker when i tried it, gpt-3 nailed ...",username when i tried it gpt3 nailed a number ...,neutral,nailed a number,person,NaN,GPT3,mathematics,NaN,NaN
1,1,1.59783e+18,@andrew___baker i just left academia (former a...,username i just left academia former assistant...,positive,NaN,NaN,NaN,GPT3,economics,"assistant lecturer, building cleaner",Educator
2,2,1.59783e+18,[gpt-3] this post on lesswrong discusses the p...,gpt3 this post on lesswrong discusses the pote...,neutral,NaN,academic,reduce,"GPT3, LessWrong",fraud detection,writer,Expert/keynote/researchers
3,3,1.59782e+18,2: for your second step in your curriculum weâ...,2 for your second step in your curriculum were...,positive,NaN,NaN,NaN,"RobertHaisfield, GPT3",work independently,NaN,NaN
4,4,1.59782e+18,<u+2728>openai gpt-3.5 series of models <u+272...,openai gpt35 series of models if youre a rese...,neutral,NaN,researcher,api,OAI API,NaN,computer scientist,Expert/keynote/researchers


In [72]:
# Drops posts w/o jobs, correctly labels target groups
education = ['lecturer', 'teacher', 'tutor', 'university teaching assistant', 'librarian', 'presenter']
ds_2.dropna(subset='job_profiles', inplace=True)

labels = []
for i in ds_2['job_profiles']:
    contained = False
    for j in education:
        if j in i:
            contained = True
    if contained:
        labels.append(1)
    else:
        labels.append(0)

In [73]:
cat_labels = []
for i in ds_2['job_profiles']:
    if education[0] in i:
        cat_labels.append('Educator')
    elif education[1] in i:
        cat_labels.append('Educator')
    elif education[2] in i:
        cat_labels.append('Expert')
    elif education[3] in i:
        cat_labels.append('Eductor')
    elif education[4] in i:
        cat_labels.append('Educator')
    elif education[5] in i:
        cat_labels.append('Expert')
    else:
        cat_labels.append('0')

In [74]:
# Adds label, drops unnecessary columns
ds_2.loc[:, 'label'] = labels
ds_2['Cat_label'] = cat_labels
ds_2.drop(columns=['Unnamed: 0', 'tweet_id','text','sentiment','tasks','users','technologies','organizations','competencies','job_profiles', 'category'], inplace=True)
ds_2.rename(columns={'label':'Label', 'original_text':'text'}, inplace=True)
ds_2

,text,Label,Cat_label
1,@andrew___baker i just left academia (former a...,1,Educator
2,[gpt-3] this post on lesswrong discusses the p...,0,0
4,<u+2728>openai gpt-3.5 series of models <u+272...,0,0
15,we further exam the quality of the generated s...,0,0
19,"for decades, many people in academia have trea...",0,0
...,...,...,...
608496,@forrrestjr @kirawontmiss some dude did this t...,1,Educator
608497,"btw, there are various pieces of news how gpt ...",0,0
608500,"rhett “mankind,” a digital artist based in aus...",0,0
608506,my professor is saying she has no issue with u...,1,Educator


In [75]:
# Drops repeated text and NaN values
ds_2.dropna(subset='text',inplace=True)
ds_2.drop_duplicates(subset='text', inplace=True)

print(ds_2['Label'].value_counts())
print(ds_2['Cat_label'].value_counts())

Label
0    25213
1    15368
Name: count, dtype: int64
Cat_label
0           25213
Educator    13981
Expert       1190
Eductor       197
Name: count, dtype: int64


In [76]:
# Combines the two datasets and saves
final_df = pd.concat([ds_1, ds_2], ignore_index=True)
final_df['Label'] = final_df['Label'].astype(int)
print(final_df['Label'].value_counts())

final_df.to_csv('final_script_csvs/dataset_1_cleaned.csv')

Label
0    26607
1    15975
Name: count, dtype: int64
